# Import libs

In [1]:
# import tensorflow and keras
import tensorflow as tf
from tensorflow import keras
from keras import layers
import keras.backend as K

# import numpy to create arrays for trial runs
import numpy as np
# matplotlib for visualisations
import matplotlib.pyplot as plt
# if using Colab, for later saving of models and loading data
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from PIL import Image
from itertools import product
import pandas as pd

Mounted at /content/drive


In [2]:
#!unzip -q /content/drive/MyDrive/BMET5933/WEEK_10/hep2imgcnn.zip -d /content/drive/MyDrive/BMET5933/WEEK_10

In [3]:
# dataset path config
train_vali_dataset_path = Path("/content/drive/MyDrive/BMET5933/WEEK_10/hep2imgcnn")

# get class names and create mapping to labels
class_names = sorted([folder.name for folder in train_vali_dataset_path.iterdir() if folder.is_dir()])
class_to_label = {class_name: idx for idx, class_name in enumerate(class_names)}
print("Class to label mapping:", class_to_label)


Class to label mapping: {'centromere': 0, 'coarse_speckled': 1, 'fine_speckled': 2, 'homogeneous': 3, 'nucleolar': 4}


In [4]:
SEED=42
BATCH_SIZE=32
IMAGE_DIR=str(train_vali_dataset_path) # this is the path to the directory containing the class subdirectories
RESCALED_IMAGE_SIZE=(51,51) # images will be all resized to this - they will have 3 channels

training_ds = keras.preprocessing.image_dataset_from_directory(
    IMAGE_DIR,
    batch_size=BATCH_SIZE,
    image_size=RESCALED_IMAGE_SIZE,
		shuffle=True,
		seed = SEED,
    validation_split=0.3,
    subset='training',
		color_mode = 'grayscale'
)

validation_ds = keras.preprocessing.image_dataset_from_directory(
	IMAGE_DIR,
	labels='inferred',
	label_mode='int',
	batch_size=BATCH_SIZE,
	image_size=RESCALED_IMAGE_SIZE,
	shuffle=True,
	seed = SEED,
	validation_split=0.3,
	subset='validation',
	color_mode = 'grayscale'
)

# you can optimise data loading with prefetching
PREFETCH_SIZE = tf.data.AUTOTUNE
training_ds = training_ds.prefetch(buffer_size=PREFETCH_SIZE)
validation_ds = validation_ds.prefetch(buffer_size=PREFETCH_SIZE)


Found 453 files belonging to 5 classes.
Using 318 files for training.
Found 453 files belonging to 5 classes.
Using 135 files for validation.


In [5]:
# define a shallow CNN model
def shallow_model(input_shape, num_classes, filter_size=5, strides=1, padding='same', num_filters=16):
	model = keras.Sequential([
		layers.Input(shape=input_shape),
		layers.Conv2D(num_filters, kernel_size=(filter_size, filter_size), activation='relu', strides=(strides, strides), padding=padding),
		layers.MaxPooling2D(pool_size=(2, 2)),
		layers.Flatten(),
		layers.Dense(num_classes, activation='softmax'),

	])
	return model

# initialise model and print summary
shallow_cnn = shallow_model(input_shape=(51, 51, 1), num_classes=len(class_names))

# check model architecture
shallow_cnn.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 51, 51, 16)     │           416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 25, 25, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 10000)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 5)              │        50,005 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 50,421 (196.96 KB)

 Trainable params: 50,421 (196.96 KB)

 Non-trainable params: 0 (0.00 B)

In [6]:
# define a deeper CNN
def deeper_model(input_shape, num_classes, filter_size=[3, 3, 3], strides=[1, 1, 1], padding = 'same'):
	model = keras.Sequential([
		layers.Input(shape=input_shape),
		layers.Conv2D(32, kernel_size=(filter_size[0], filter_size[0]), activation='relu', strides=(strides[0], strides[0]), padding=padding),
		layers.MaxPooling2D(pool_size=(2, 2)),
		layers.Conv2D(64, kernel_size=(filter_size[1], filter_size[1]), activation='relu', strides=(strides[1], strides[1]), padding=padding),
		layers.MaxPooling2D(pool_size=(2, 2)),
		layers.Conv2D(128, kernel_size=(filter_size[2], filter_size[2]), activation='relu', strides=(strides[2], strides[2]), padding=padding),
		layers.MaxPooling2D(pool_size=(2, 2)),
		layers.Flatten(),
		layers.Dense(64, activation='relu'),
		layers.Dense(num_classes, activation='softmax')
	])
	return model

# initialise deeper model and print summary
deeper_cnn = deeper_model(input_shape=(51, 51, 1), num_classes=len(class_names))

# check deeper model architecture
deeper_cnn.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_1 (Conv2D)               │ (None, 51, 51, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 25, 25, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 25, 25, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 12, 12, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 12, 12, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │       294,976 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 387,973 (1.48 MB)

 Trainable params: 387,973 (1.48 MB)

 Non-trainable params: 0 (0.00 B)

In [7]:
LEARNING_RATE = 1e-4 # Some common values are 1e-3 (0.001) or 1e-5 (0.00001)
BATCH_SIZE = 32 # train with this many images per iteration [5, 10, or 20 might be good for a trial run]
NUM_EPOCHS = 25 # how many epochs to train for (each epoch visits the training data once) [5 might be good for a trial run]

def train_model(model, training_ds, validation_ds, lr, epochs):
	# model training config
	model.compile(
		optimizer=keras.optimizers.Adam(lr),
		loss="sparse_categorical_crossentropy",
		metrics=["accuracy"]
	)
	training_history = model.fit(
		training_ds,
		validation_data=validation_ds,
		epochs=epochs
	)
	return training_history

# 1. train shallow CNN
# train_history will store the metrics for each epoch, for use in generating graphs
shallow_train = train_model(shallow_cnn, training_ds, validation_ds, LEARNING_RATE, NUM_EPOCHS)

Epoch 1/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 140s 14s/step - accuracy: 0.2170 - loss: 8.8730 - val_accuracy: 0.3926 - val_loss: 3.8829
Epoch 2/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 115ms/step - accuracy: 0.3208 - loss: 4.6097 - val_accuracy: 0.4667 - val_loss: 2.9492
Epoch 3/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 107ms/step - accuracy: 0.3648 - loss: 3.1959 - val_accuracy: 0.4444 - val_loss: 2.5558
Epoch 4/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 151ms/step - accuracy: 0.4057 - loss: 2.4393 - val_accuracy: 0.4815 - val_loss: 2.0037
Epoch 5/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 166ms/step - accuracy: 0.5000 - loss: 2.0906 - val_accuracy: 0.4593 - val_loss: 2.7518
Epoch 6/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 192ms/step - accuracy: 0.4843 - loss: 2.2456 - val_accuracy: 0.4963 - val_loss: 2.5463
Epoch 7/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 114ms/step - accuracy: 0.5377 - loss: 2.0500 - val_accuracy: 0.4741 - val_loss: 1.7168
Epoch 8/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 112ms/step - accuracy: 0.5503 - loss: 1.4932 - val_accuracy: 0.

In [8]:
# 2. train deeper CNN
deeper_train = train_model(deeper_cnn, training_ds, validation_ds, LEARNING_RATE, NUM_EPOCHS)

Epoch 1/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 282ms/step - accuracy: 0.2075 - loss: 3.5277 - val_accuracy: 0.2741 - val_loss: 2.4863
Epoch 2/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 228ms/step - accuracy: 0.2736 - loss: 2.0686 - val_accuracy: 0.2000 - val_loss: 2.1084
Epoch 3/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 191ms/step - accuracy: 0.3208 - loss: 1.5328 - val_accuracy: 0.3630 - val_loss: 1.4944
Epoch 4/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 197ms/step - accuracy: 0.4308 - loss: 1.3382 - val_accuracy: 0.3407 - val_loss: 1.3965
Epoch 5/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 269ms/step - accuracy: 0.5063 - loss: 1.1796 - val_accuracy: 0.4296 - val_loss: 1.2848
Epoch 6/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 308ms/step - accuracy: 0.5566 - loss: 1.0967 - val_accuracy: 0.5333 - val_loss: 1.2450
Epoch 7/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 228ms/step - accuracy: 0.6195 - loss: 1.0109 - val_accuracy: 0.5704 - val_loss: 1.1643
Epoch 8/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 193ms/step - accuracy: 0.6478 - loss: 0.9673 - val_accuracy: 0.

In [9]:
test_dataset_path = Path('/content/drive/MyDrive/BMET5933/WEEK_10/hep2img_testset')

def read_img(img_path):
  # read image and convert into np array
  img = np.array(Image.open(img_path))

  # expand channel dim
  if img.ndim == 2:
    img = img[..., np.newaxis]
  img = img / 255.0
  img = tf.image.resize(img, (51, 51))
  return img

def model_predict(model, image, class_names: list):
  image = np.array(image)

  if image.ndim == 2:
    # expand channel dim and norm
    image = np.expand_dims(image, axis = 0)
    image = image / 255.0
    image = tf.image.resize(51, 51)

  # predict image label
  prediction = model.predict(image)
  label = np.argmax(prediction, axis = 1)
  return label

def evaluate_model(model, test_ds, class_names: list):
  # get true labels and predicted labels
  true_labels = []
  pred_labels = []
  for image, label in test_ds:
    true_labels.extend(label.numpy())
    pred_label = model_predict(model, image, class_names)
    pred_labels.extend(pred_label)
    
  # calculate accuracy
  correct = sum(p == t for p, t in zip(pred_labels, true_labels))
  total = len(true_labels)
  accuracy = correct / total
  return accuracy


In [10]:
# create test dataset
test_ds = keras.preprocessing.image_dataset_from_directory(
	str(test_dataset_path),
	labels='inferred',
	label_mode='int',
	batch_size=BATCH_SIZE,
	image_size=RESCALED_IMAGE_SIZE,
	shuffle=False,
	color_mode = 'grayscale'
)

# evaluate shallow CNN on test set
shallow_accuracy  = evaluate_model(shallow_cnn, test_ds, class_names)
print(f"Shallow CNN Test Accuracy: {shallow_accuracy:.4f}")

#evaluate deeper CNN on test set
deeper_accuracy = evaluate_model(deeper_cnn, test_ds, class_names)
print(f"Deeper CNN Test Accuracy: {deeper_accuracy:.4f}")


Found 25 files belonging to 5 classes.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
Shallow CNN Test Accuracy: 0.8000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 123ms/step
Deeper CNN Test Accuracy: 0.8800


# Challenge part:

## 1.replace maxpool with avgpool

In [11]:
# use avg pooling
def avgpool_shallow_model(input_shape, num_classes):
	model = keras.Sequential([
		layers.Input(shape=input_shape),
		layers.Conv2D(16, kernel_size=(5, 5), activation='relu'),
		layers.AveragePooling2D(pool_size=(2, 2)),
		layers.Flatten(),
		layers.Dense(num_classes, activation='softmax'),
	])
	return model

avg_pool_model = avgpool_shallow_model(input_shape=(51, 51, 1), num_classes=len(class_names))

# train avg pooling model
avg_pool_model_training = train_model(avg_pool_model, training_ds, validation_ds, LEARNING_RATE, NUM_EPOCHS)

Epoch 1/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 118ms/step - accuracy: 0.1918 - loss: 8.2646 - val_accuracy: 0.1926 - val_loss: 4.9673
Epoch 2/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 105ms/step - accuracy: 0.2421 - loss: 5.1429 - val_accuracy: 0.3259 - val_loss: 4.8739
Epoch 3/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 101ms/step - accuracy: 0.2736 - loss: 4.1472 - val_accuracy: 0.2963 - val_loss: 4.1062
Epoch 4/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 101ms/step - accuracy: 0.3082 - loss: 3.8215 - val_accuracy: 0.2593 - val_loss: 3.7946
Epoch 5/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 119ms/step - accuracy: 0.3553 - loss: 3.3404 - val_accuracy: 0.3481 - val_loss: 4.1310
Epoch 6/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 179ms/step - accuracy: 0.3396 - loss: 3.0787 - val_accuracy: 0.4370 - val_loss: 2.6770
Epoch 7/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 153ms/step - accuracy: 0.3648 - loss: 2.5117 - val_accuracy: 0.3630 - val_loss: 2.8577
Epoch 8/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 105ms/step - accuracy: 0.4245 - loss: 2.2089 - val_accuracy: 0.

### conclusion:
from the training history of avg_pool_model, we can see the degradation of model due to changing pooling layer.

## 2.Parrameter changing

## Parameter changing note:
All the parameter changes are based on original shallow CNN, including:
1. kernel_size 5 -> 7;
2. stride 1 -> 2;
3. padding method same -> valid

In [12]:
# model variations with config
# filter size 7
shallow_model_filter7 = shallow_model(input_shape=(51, 51, 1), num_classes=len(class_names), filter_size=7)
filter7_train = train_model(shallow_model_filter7, training_ds, validation_ds, LEARNING_RATE, NUM_EPOCHS)


Epoch 1/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 143ms/step - accuracy: 0.1635 - loss: 12.2586 - val_accuracy: 0.1778 - val_loss: 6.9262
Epoch 2/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 124ms/step - accuracy: 0.2138 - loss: 4.7395 - val_accuracy: 0.2444 - val_loss: 3.5813
Epoch 3/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 193ms/step - accuracy: 0.2893 - loss: 3.4112 - val_accuracy: 0.2741 - val_loss: 3.4433
Epoch 4/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 224ms/step - accuracy: 0.3711 - loss: 2.6437 - val_accuracy: 0.3778 - val_loss: 2.7694
Epoch 5/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 137ms/step - accuracy: 0.4560 - loss: 2.1835 - val_accuracy: 0.4444 - val_loss: 2.1131
Epoch 6/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 124ms/step - accuracy: 0.4654 - loss: 1.9120 - val_accuracy: 0.3556 - val_loss: 2.3388
Epoch 7/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 128ms/step - accuracy: 0.5094 - loss: 1.8353 - val_accuracy: 0.4074 - val_loss: 2.1978
Epoch 8/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 130ms/step - accuracy: 0.5283 - loss: 1.5888 - val_accuracy: 0

In [13]:
# stride 2
shallow_model_stride2 = shallow_model(input_shape=(51, 51, 1), num_classes=len(class_names), strides=2)
stride2_train = train_model(shallow_model_stride2, training_ds, validation_ds, LEARNING_RATE, NUM_EPOCHS)

Epoch 1/25


10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 136ms/step - accuracy: 0.1604 - loss: 9.0625 - val_accuracy: 0.1630 - val_loss: 6.8553
Epoch 2/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 110ms/step - accuracy: 0.1730 - loss: 7.3066 - val_accuracy: 0.1926 - val_loss: 6.3180
Epoch 3/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 115ms/step - accuracy: 0.1981 - loss: 6.2169 - val_accuracy: 0.1852 - val_loss: 6.5945
Epoch 4/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 103ms/step - accuracy: 0.2296 - loss: 5.4309 - val_accuracy: 0.2444 - val_loss: 5.5173
Epoch 5/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 92ms/step - accuracy: 0.2201 - loss: 5.0197 - val_accuracy: 0.2222 - val_loss: 5.1582
Epoch 6/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 89ms/step - accuracy: 0.2421 - loss: 4.4092 - val_accuracy: 0.2222 - val_loss: 5.1154
Epoch 7/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 89ms/step - accuracy: 0.2925 - loss: 4.0774 - val_accuracy: 0.2741 - val_loss: 4.5355
Epoch 8/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 92ms/step - accuracy: 0.3082 - loss: 3.6506 - val_accuracy: 0.3037 - val_loss

In [14]:
# valid padding
shallow_model_padding_valid = shallow_model(input_shape=(51, 51, 1), num_classes=len(class_names), padding='valid')
padding_valid_train = train_model(shallow_model_padding_valid, training_ds, validation_ds, LEARNING_RATE, NUM_EPOCHS)


Epoch 1/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 123ms/step - accuracy: 0.2044 - loss: 13.0162 - val_accuracy: 0.1556 - val_loss: 8.0945
Epoch 2/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 152ms/step - accuracy: 0.1981 - loss: 10.4120 - val_accuracy: 0.1259 - val_loss: 5.8343
Epoch 3/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 150ms/step - accuracy: 0.2107 - loss: 8.0197 - val_accuracy: 0.3111 - val_loss: 5.7962
Epoch 4/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 179ms/step - accuracy: 0.2453 - loss: 5.8318 - val_accuracy: 0.2667 - val_loss: 6.4059
Epoch 5/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 110ms/step - accuracy: 0.2579 - loss: 5.5586 - val_accuracy: 0.2222 - val_loss: 5.4710
Epoch 6/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 109ms/step - accuracy: 0.2642 - loss: 4.4085 - val_accuracy: 0.2296 - val_loss: 4.1268
Epoch 7/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 104ms/step - accuracy: 0.3208 - loss: 3.5347 - val_accuracy: 0.2741 - val_loss: 3.4769
Epoch 8/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 105ms/step - accuracy: 0.3522 - loss: 3.4366 - val_accuracy: 

In [15]:
# number of filters 32
shallow_model_filters32 = shallow_model(input_shape=(51, 51, 1), num_classes=len(class_names), num_filters=32)
filters32_train = train_model(shallow_model_filters32, training_ds, validation_ds, LEARNING_RATE, NUM_EPOCHS)


Epoch 1/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 137ms/step - accuracy: 0.2044 - loss: 9.7368 - val_accuracy: 0.1704 - val_loss: 7.8405
Epoch 2/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 123ms/step - accuracy: 0.2390 - loss: 5.3707 - val_accuracy: 0.2074 - val_loss: 3.4004
Epoch 3/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 111ms/step - accuracy: 0.2987 - loss: 3.3096 - val_accuracy: 0.2889 - val_loss: 2.4295
Epoch 4/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 122ms/step - accuracy: 0.3836 - loss: 2.2690 - val_accuracy: 0.3704 - val_loss: 3.0003
Epoch 5/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 117ms/step - accuracy: 0.3994 - loss: 2.3220 - val_accuracy: 0.3481 - val_loss: 2.7510
Epoch 6/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 111ms/step - accuracy: 0.4654 - loss: 2.0464 - val_accuracy: 0.4667 - val_loss: 2.0416
Epoch 7/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 182ms/step - accuracy: 0.4497 - loss: 1.6884 - val_accuracy: 0.4148 - val_loss: 2.2692
Epoch 8/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 175ms/step - accuracy: 0.4717 - loss: 1.5261 - val_accuracy: 0.

In [16]:
# evaluate model variations on test set
filter7_accuracy = evaluate_model(shallow_model_filter7, test_ds, class_names)
print(f"Shallow CNN with filter size 7 Test Accuracy: {filter7_accuracy:.4f}")

filters32_accuracy = evaluate_model(shallow_model_filters32, test_ds, class_names)
print(f"Shallow CNN with 32 filters Test Accuracy: {filters32_accuracy:.4f}")

stride2_accuracy = evaluate_model(shallow_model_stride2, test_ds, class_names)
print(f"Shallow CNN with stride 2 Test Accuracy: {stride2_accuracy:.4f}")

padding_valid_accuracy = evaluate_model(shallow_model_padding_valid, test_ds, class_names)
print(f"Shallow CNN with valid padding Test Accuracy: {padding_valid_accuracy:.4f}")

challenge2_results = pd.DataFrame([
	{"change": "baseline", "filter_size": 5, "num_filters": 16, "stride": 1, "padding": "same", "test_accuracy": shallow_accuracy},
	{"change": "filter size 7", "filter_size": 7, "num_filters": 16, "stride": 1, "padding": "same", "test_accuracy": filter7_accuracy},
	{"change": "32 filters", "filter_size": 5, "num_filters": 32, "stride": 1, "padding": "same", "test_accuracy": filters32_accuracy},
	{"change": "stride 2", "filter_size": 5, "num_filters": 16, "stride": 2, "padding": "same", "test_accuracy": stride2_accuracy},
	{"change": "valid padding", "filter_size": 5, "num_filters": 16, "stride": 1, "padding": "valid", "test_accuracy": padding_valid_accuracy},
])

challenge2_results["accuracy_change"] = challenge2_results["test_accuracy"] - shallow_accuracy
challenge2_results


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step
Shallow CNN with filter size 7 Test Accuracy: 0.7600
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
Shallow CNN with 32 filters Test Accuracy: 0.8800


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step
Shallow CNN with stride 2 Test Accuracy: 0.5200


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step
Shallow CNN with valid padding Test Accuracy: 0.6000


,change,filter_size,num_filters,stride,padding,test_accuracy,accuracy_change
0,baseline,5,16,1,same,0.80,0.00
1,filter size 7,7,16,1,same,0.76,-0.04
2,32 filters,5,32,1,same,0.88,0.08
3,stride 2,5,16,2,same,0.52,-0.28
4,valid padding,5,16,1,valid,0.60,-0.20


## Challenge 3: Changing training hyperparameters

In [ ]:
# challenge 3 - learning rate and epochs
lr_variations = [1e-3, 1e-4]
epoch_variations = [15, 25]

# train model with learning rate 1e-3 and original epochs
lr_variation_training = train_model(shallow_cnn, training_ds, validation_ds, lr=1e-3, epochs=NUM_EPOCHS)
#evaluate model with learning rate 1e-3
lr_variation_accuracy = evaluate_model(shallow_cnn, test_ds, class_names)
print(f"Shallow CNN Test Accuracy with learning rate 1e-3: {lr_variation_accuracy:.4f}")


Epoch 1/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 187ms/step - accuracy: 0.3208 - loss: 44.2864 - val_accuracy: 0.2667 - val_loss: 23.9010
Epoch 2/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 114ms/step - accuracy: 0.3145 - loss: 15.2873 - val_accuracy: 0.3333 - val_loss: 6.6919
Epoch 3/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 116ms/step - accuracy: 0.4025 - loss: 4.2653 - val_accuracy: 0.4222 - val_loss: 1.6894
Epoch 4/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 111ms/step - accuracy: 0.5786 - loss: 1.0859 - val_accuracy: 0.4667 - val_loss: 1.5504
Epoch 5/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 115ms/step - accuracy: 0.5912 - loss: 1.0193 - val_accuracy: 0.5037 - val_loss: 1.3950
Epoch 6/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 115ms/step - accuracy: 0.6509 - loss: 0.8796 - val_accuracy: 0.5630 - val_loss: 1.2971
Epoch 7/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 111ms/step - accuracy: 0.6792 - loss: 0.7821 - val_accuracy: 0.5556 - val_loss: 1.1793
Epoch 8/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 116ms/step - accuracy: 0.7170 - loss: 0.7474 - val_accuracy:

In [20]:
# train model with original learning rate and 15 epochs
epoch_variation_training = train_model(shallow_cnn, training_ds, validation_ds, lr=LEARNING_RATE, epochs=15)

# evaluate model with 15 epochs
epoch_variation_accuracy = evaluate_model(shallow_cnn, test_ds, class_names)
print(f"Shallow CNN Test Accuracy with 15 epochs: {epoch_variation_accuracy:.4f}")

Epoch 1/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 218ms/step - accuracy: 0.9119 - loss: 0.3172 - val_accuracy: 0.5852 - val_loss: 1.2099
Epoch 2/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 154ms/step - accuracy: 0.9025 - loss: 0.3136 - val_accuracy: 0.5704 - val_loss: 1.1921
Epoch 3/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 108ms/step - accuracy: 0.9214 - loss: 0.2983 - val_accuracy: 0.5852 - val_loss: 1.2239
Epoch 4/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 111ms/step - accuracy: 0.9340 - loss: 0.2840 - val_accuracy: 0.5852 - val_loss: 1.2141
Epoch 5/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 109ms/step - accuracy: 0.9245 - loss: 0.2830 - val_accuracy: 0.5852 - val_loss: 1.2181
Epoch 6/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 113ms/step - accuracy: 0.9434 - loss: 0.2665 - val_accuracy: 0.5926 - val_loss: 1.1836
Epoch 7/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 113ms/step - accuracy: 0.9277 - loss: 0.2675 - val_accuracy: 0.5852 - val_loss: 1.1928
Epoch 8/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 112ms/step - accuracy: 0.9403 - loss: 0.2555 - val_accuracy: 0.

## Challenge 4: Deeper model

In [21]:
def even_deeper_model(input_shape, num_classes):
	model = keras.Sequential([
		layers.Input(shape=input_shape),
		layers.Conv2D(32, kernel_size=(5, 5), activation='relu', padding='same'),
		layers.MaxPooling2D(pool_size=(2, 2)),
		layers.Conv2D(64, kernel_size=(5, 5), activation='relu', padding='same'),
		layers.MaxPooling2D(pool_size=(2, 2)),
		layers.Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same'),
		layers.MaxPooling2D(pool_size=(2, 2)),
		layers.Conv2D(256, kernel_size=(3, 3), activation='relu', padding='same'),
		layers.MaxPooling2D(pool_size=(2, 2)),
		layers.Flatten(),
		layers.Dense(128, activation='relu'),
		layers.Dense(64, activation='relu'),
		layers.Dense(num_classes, activation='softmax')
	])
	return model

even_deeper_cnn = even_deeper_model(input_shape=(51, 51, 1), num_classes=len(class_names))
even_deeper_cnn.summary()

Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_13 (Conv2D)              │ (None, 51, 51, 32)     │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_12 (MaxPooling2D) │ (None, 25, 25, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_14 (Conv2D)              │ (None, 25, 25, 64)     │        51,264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_13 (MaxPooling2D) │ (None, 12, 12, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_15 (Conv2D)              │ (None, 12, 12, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_14 (MaxPooling2D) │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_16 (Conv2D)              │ (None, 6, 6, 256)      │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_15 (MaxPooling2D) │ (None, 3, 3, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_8 (Flatten)             │ (None, 2304)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 128)            │       295,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 724,741 (2.76 MB)

 Trainable params: 724,741 (2.76 MB)

 Non-trainable params: 0 (0.00 B)

In [19]:
even_deeper_train = train_model(even_deeper_cnn, training_ds, validation_ds, LEARNING_RATE, NUM_EPOCHS)
even_deeper_accuracy = evaluate_model(even_deeper_cnn, test_ds, class_names)
print(f"Even deeper CNN Test Accuracy: {even_deeper_accuracy:.4f}")

Epoch 1/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 8s 479ms/step - accuracy: 0.2264 - loss: 2.5368 - val_accuracy: 0.2444 - val_loss: 1.5062
Epoch 2/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 4s 360ms/step - accuracy: 0.3585 - loss: 1.4592 - val_accuracy: 0.3852 - val_loss: 1.4346
Epoch 3/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 4s 350ms/step - accuracy: 0.4780 - loss: 1.2343 - val_accuracy: 0.4000 - val_loss: 1.2411
Epoch 4/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 550ms/step - accuracy: 0.5692 - loss: 1.1142 - val_accuracy: 0.4741 - val_loss: 1.1507
Epoch 5/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 8s 348ms/step - accuracy: 0.6006 - loss: 1.0276 - val_accuracy: 0.5259 - val_loss: 1.0732
Epoch 6/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 6s 589ms/step - accuracy: 0.6541 - loss: 0.9094 - val_accuracy: 0.5111 - val_loss: 1.0529
Epoch 7/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 4s 369ms/step - accuracy: 0.6195 - loss: 0.9288 - val_accuracy: 0.5852 - val_loss: 1.0417
Epoch 8/25
10/10 ━━━━━━━━━━━━━━━━━━━━ 4s 351ms/step - accuracy: 0.7075 - loss: 0.8246 - val_accuracy: 0.